In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, matthews_corrcoef
from kneed import KneeLocator

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [2]:
phase2_df1=pd.read_csv('/home/imokhtatif/.vscode-server/Chlamy_Project_v2-main/Data/2025_5_phase2.csv',low_memory=False)

In [3]:
from scipy import interpolate
from scipy.stats import rankdata
def normalize_quantiles(A, ties=False):
    A = np.asarray(A, dtype=np.float64)
    n_rows, n_cols = A.shape
    if n_cols == 1:
        return A.copy()

    i = np.linspace(0, 1, n_rows)
    S = np.full((n_rows, n_cols), np.nan)
    nobs = np.zeros(n_cols, dtype=int)
    sort_idx = []

    for j in range(n_cols):
        col = A[:, j]
        not_nan = ~np.isnan(col)
        x = col[not_nan]
        nobs[j] = len(x)
        sort_order = np.argsort(x)
        sorted_x = x[sort_order]

        if nobs[j] < n_rows:
            f = interpolate.interp1d(np.linspace(0, 1, nobs[j]), sorted_x,
                                     bounds_error=False, fill_value="extrapolate")
            S[:, j] = f(i)
        else:
            S[:, j] = sorted_x

        sort_idx.append(np.argsort(np.argsort(col[not_nan])))

    m = np.nanmean(S, axis=1)
    A_out = np.full_like(A, np.nan)

    for j in range(n_cols):
        col = A[:, j]
        not_nan = ~np.isnan(col)

        if ties:
            r = rankdata(col[not_nan], method='average')
            quant_pos = (r - 1) / (nobs[j] - 1)
            f = interpolate.interp1d(i, m, bounds_error=False, fill_value="extrapolate")
            A_out[not_nan, j] = f(quant_pos)
        else:
            ranks = sort_idx[j]
            A_out[not_nan, j] = m[ranks.astype(int)]

    return A_out

In [4]:
# Step 1: Filter data for the three plates
plates = ['32v1', '32v2', '32v3']
df_30v =phase2_df1[phase2_df1['plate'].isin(plates)]

# Step 2: Count rows per (plate, mutant_ID, mutated_genes, light_regime)
group_counts = (
    df_30v.groupby(['plate', 'light_regime', 'mutant_ID', 'mutated_genes'])
    .size()
    .reset_index(name='count')
)

# Step 3: For each plate and light_regime, count how many mutants had 1, 2, ... rows
summary = (
    group_counts.groupby(['light_regime','plate', 'count'])
    .size()
    .reset_index(name='n_mutants')
)

# Optional: Sort for easier reading
summary = summary.sort_values(by=['n_mutants','light_regime','plate', 'count'])

# Show result
summary

,light_regime,plate,count,n_mutants
2,10min-10min,32v1,6,1
5,10min-10min,32v2,6,1
8,10min-10min,32v3,6,1
11,1min-1min,32v1,6,1
14,1min-1min,32v2,6,1
...,...,...,...,...
39,20h_ML,32v2,1,367
42,20h_ML,32v3,1,367
48,2h-2h,32v2,1,367
66,5min-5min,32v2,1,367


### plate 32 20_ML

In [5]:
def quantile_normalize_light_regime(df, light_regime, plates, y2_cols, tie_handling=False):
    """
    Quantile-normalize all y2_cols across selected plates within a given light regime.
    
    Parameters:
    - df: pandas DataFrame, full dataset
    - light_regime: str, target light regime (e.g. '20h_ML')
    - plates: list of str, target plate names (e.g. ['30v1', '30v2', '30v3'])
    - y2_cols: list of str, column names like ['y2_1', ..., 'y2_44']
    - tie_handling: bool, passed to normalize_quantiles (default=True)
    
    Returns:
    - df_normalized: pandas DataFrame with normalized y2_cols
    """
    # Filter data
    subset_df = df[(df['light_regime'] == light_regime) & (df['plate'].isin(plates))].copy()
    df_normalized = subset_df.copy()

    for timepoint in y2_cols:
        position_values = []
        valid_plate_indices = {}

        for plate in plates:
            plate_df = subset_df[subset_df['plate'] == plate].copy()

            wt_rows = plate_df[plate_df['mutant_ID'] == 'WT'].copy()
            non_wt_rows = plate_df[plate_df['mutant_ID'] != 'WT'].copy()

            wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'well_id'])
            non_wt_rows = non_wt_rows.sort_values(['mutant_ID', 'mutated_genes'])

            sorted_df = pd.concat([wt_rows, non_wt_rows], axis=0)
            values = sorted_df[timepoint].values
            index = sorted_df.index.values

            position_values.append(values)
            valid_plate_indices[plate] = index

        # Validate shape
        lengths = [len(v) for v in position_values]
        if len(set(lengths)) != 1:
            raise ValueError(f"Length mismatch at {timepoint}: {lengths}")

        matrix = np.column_stack(position_values)
        normalized_matrix = normalize_quantiles(matrix, ties=tie_handling)

        # Write back
        for col_idx, plate in enumerate(plates):
            indices = valid_plate_indices[plate]
            df_normalized.loc[indices, timepoint] = normalized_matrix[:, col_idx]

    return df_normalized

In [6]:
# Define inputs
plates = ['32v1', '32v2', '32v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]

# Run normalization
phase2_32_20h_ML_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='20h_ML',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_32_20h_ML_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
19719,32v1,LMJ.RY0402.146061,Cre02.g115600,P24,0.208805,0.267782,0.291203,0.301460,0.305910,0.276875,...,0.265624,0.251320,0.249690,0.226460,0.263906,0.230052,0.250185,0.234120,0.223986,0.244580
19720,32v1,LMJ.RY0402.048478,Cre06.g270450,P20,0.341395,0.397976,0.376696,0.388874,0.354813,0.381043,...,0.396842,0.357797,0.396474,0.314284,0.349169,0.305581,0.345400,0.357273,0.333408,0.364768
19721,32v1,LMJ.RY0402.210464,Cre02.g077951,K20,0.311715,0.322834,0.346912,0.350660,0.362679,0.372225,...,0.305005,0.289180,0.292143,0.295937,0.246723,0.292876,0.245409,0.262818,0.289936,0.242976
19722,32v1,LMJ.RY0402.210458,"Cre17.g744997,Cre01.g034250",K19,0.329590,0.366730,0.344731,0.329335,0.332540,0.338511,...,0.293999,0.286706,0.312700,0.318298,0.295387,0.278866,0.306182,0.282238,0.294486,0.269703
19723,32v1,LMJ.RY0402.209208,Cre06.g256300,K18,0.093859,0.097177,0.100021,0.193270,0.176433,0.161896,...,0.176997,0.101872,0.064737,0.151549,0.082802,0.123352,0.123380,0.109612,0.141286,0.111623
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31097,32v3,LMJ.RY0402.215150,Cre17.g724650,K21,0.365138,0.355368,0.368891,0.376068,0.397376,0.402794,...,0.337018,0.357797,0.328567,0.326779,0.343451,0.334395,0.370300,0.341599,0.324724,0.317108
31098,32v3,LMJ.RY0402.173502,Cre06.g278219,K22,0.401359,0.395604,0.426147,0.430300,0.419204,0.410436,...,0.358904,0.366833,0.391656,0.356272,0.386093,0.369292,0.366952,0.357273,0.363823,0.352961
31099,32v3,LMJ.RY0402.077964,Cre01.g019550,K23,0.208805,0.193739,0.251620,0.223936,0.274272,0.266623,...,0.221323,0.220079,0.222588,0.212766,0.256772,0.237953,0.248277,0.229753,0.243393,0.243781
31100,32v3,LMJ.RY0402.107867,Cre17.g704400,K24,0.325682,0.348594,0.344454,0.313894,0.362679,0.337168,...,0.326473,0.311664,0.285029,0.311060,0.338749,0.331021,0.305380,0.292661,0.341220,0.308704


In [7]:
plates = ['32v1', '32v2','32v3']
phase2_32_20h_ML= phase2_df1[(phase2_df1['light_regime'] == '20h_ML') & (phase2_df1['plate'].isin(plates))].copy()
phase2_32_20h_ML[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols[:10]]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,y2_7,y2_8,y2_9,y2_10
19719,32v1,LMJ.RY0402.146061,Cre02.g115600,P24,0.201459,0.270127,0.284102,0.305213,0.297936,0.261904,0.334571,0.300440,0.316412,0.308160
19720,32v1,LMJ.RY0402.048478,Cre06.g270450,P20,0.333859,0.392427,0.372982,0.380526,0.338695,0.374140,0.394284,0.359182,0.334045,0.431763
19721,32v1,LMJ.RY0402.210464,Cre02.g077951,K20,0.303585,0.319703,0.342165,0.344543,0.349481,0.366270,0.359228,0.339114,0.342496,0.319021
19722,32v1,LMJ.RY0402.210458,"Cre17.g744997,Cre01.g034250",K19,0.323043,0.366751,0.337821,0.325072,0.319421,0.331907,0.331559,0.329654,0.302691,0.346568
19723,32v1,LMJ.RY0402.209208,Cre06.g256300,K18,0.078591,0.089459,0.067118,0.157712,0.120825,0.118087,0.128841,0.119648,0.114171,0.125721
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31097,32v3,LMJ.RY0402.215150,Cre17.g724650,K21,0.374789,0.360604,0.371988,0.390428,0.411013,0.410698,0.436304,0.442273,0.403196,0.400158
31098,32v3,LMJ.RY0402.173502,Cre06.g278219,K22,0.411550,0.406307,0.431847,0.442512,0.436995,0.419308,0.456203,0.433083,0.412389,0.403420
31099,32v3,LMJ.RY0402.077964,Cre01.g019550,K23,0.219216,0.239437,0.248607,0.274697,0.287499,0.277060,0.289123,0.291483,0.251553,0.295409
31100,32v3,LMJ.RY0402.107867,Cre17.g704400,K24,0.333807,0.354778,0.348867,0.329779,0.375337,0.343581,0.372514,0.402801,0.341513,0.329711


### plate 32 20_HL

In [8]:
# Define inputs
plates = ['32v1', '32v2', '32v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]

# Run normalization
phase2_32_20h_HL_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='20h_HL',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_32_20h_HL_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
19342,32v1,LMJ.RY0402.040792,Cre02.g098750,A06,0.097818,0.113840,0.114692,0.098696,0.129448,0.090323,...,-0.035862,0.066276,0.047889,0.012459,0.017739,0.002141,0.024301,0.056465,0.044999,0.007785
19343,32v1,LMJ.RY0402.210754,"Cre06.g263357,Cre10.g433100",K22,0.236664,0.226790,0.257425,0.261719,0.268060,0.261001,...,0.123102,0.166663,0.145651,0.152570,0.134833,0.187938,0.097403,0.134396,0.177519,0.169587
19344,32v1,LMJ.RY0402.210603,Cre02.g087450,K21,0.195673,0.207985,0.152399,0.161358,0.199803,0.201654,...,0.113861,0.074049,0.099279,0.074335,0.118406,0.086189,0.093081,0.139870,0.117372,0.128010
19345,32v1,LMJ.RY0402.210464,Cre02.g077951,K20,0.143327,0.174001,0.156769,0.158757,0.150638,0.135934,...,0.090250,0.059732,0.084775,0.034980,0.077181,0.061406,0.091856,0.079553,0.037741,0.037658
19346,32v1,LMJ.RY0402.210458,"Cre17.g744997,Cre01.g034250",K19,0.211431,0.201929,0.177083,0.208156,0.210912,0.080067,...,0.085902,0.024071,0.071452,0.003090,0.091418,0.081360,0.065684,0.092545,0.030290,0.062347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30720,32v3,LMJ.RY0402.184490,Cre02.g076200,L23,0.152671,0.156480,0.120531,0.090130,0.150210,0.129212,...,0.081126,0.069474,0.018671,0.056889,0.103520,0.012305,0.065684,0.049025,0.044999,0.062093
30721,32v3,LMJ.RY0402.142353,Cre06.g295050,K19,0.314752,0.324856,0.254929,0.287496,0.293839,0.194799,...,0.062303,0.104112,0.104489,0.127845,0.081973,0.059274,0.044750,0.070875,0.082729,0.062347
30722,32v3,LMJ.RY0402.253374,Cre02.g077951,K18,0.223769,0.232503,0.259495,0.236463,0.213830,0.205153,...,0.146024,0.143436,0.116228,0.113465,0.119462,0.138561,0.136071,0.141224,0.111149,0.129420
30723,32v3,LMJ.RY0402.172376,Cre06.g278219,K17,0.183962,0.143873,0.124908,0.143832,0.160210,0.120364,...,0.102461,0.094087,0.052457,0.091412,0.060515,0.086743,0.102909,0.084603,0.111756,0.077387


In [9]:
plates = ['32v1', '32v2','32v3']
phase2_32_20h_HL= phase2_df1[(phase2_df1['light_regime'] == '20h_HL') & (phase2_df1['plate'].isin(plates))].copy()
phase2_32_20h_HL[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols[:10]]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,y2_7,y2_8,y2_9,y2_10
19342,32v1,LMJ.RY0402.040792,Cre02.g098750,A06,0.073017,0.096578,0.092203,0.066138,0.111367,0.065858,0.017939,0.053192,0.041920,0.110379
19343,32v1,LMJ.RY0402.210754,"Cre06.g263357,Cre10.g433100",K22,0.223561,0.212659,0.241239,0.238167,0.252680,0.240941,0.212680,0.252194,0.184284,0.163150
19344,32v1,LMJ.RY0402.210603,Cre02.g087450,K21,0.182431,0.195138,0.133831,0.136200,0.183265,0.179556,0.202271,0.210011,0.170084,0.165390
19345,32v1,LMJ.RY0402.210464,Cre02.g077951,K20,0.127866,0.160238,0.139538,0.133231,0.128644,0.116749,0.150405,0.105204,0.116738,0.133027
19346,32v1,LMJ.RY0402.210458,"Cre17.g744997,Cre01.g034250",K19,0.195602,0.189034,0.160856,0.183785,0.194799,0.054554,0.147272,0.177431,0.133208,0.069073
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30720,32v3,LMJ.RY0402.184490,Cre02.g076200,L23,0.158056,0.168785,0.139301,0.118765,0.162887,0.152355,0.202004,0.133455,0.131888,0.097923
30721,32v3,LMJ.RY0402.142353,Cre06.g295050,K19,0.312174,0.326765,0.257048,0.299310,0.300779,0.213426,0.256351,0.233545,0.263102,0.256689
30722,32v3,LMJ.RY0402.253374,Cre02.g077951,K18,0.223949,0.238014,0.263084,0.251092,0.220224,0.224036,0.260743,0.254936,0.221333,0.243406
30723,32v3,LMJ.RY0402.172376,Cre06.g278219,K17,0.183094,0.157133,0.142048,0.160823,0.172158,0.143267,0.155972,0.175817,0.157497,0.133510


### 32 plate 2h-2h


In [10]:
# Define inputs
plates = ['32v1', '32v2']
y2_cols = [f'y2_{i}' for i in range(1, 49)]

# Get only mutants present in all plates
plates_of_interest = plates  # or your list of plates
subset = phase2_df1[
    (phase2_df1['light_regime'] == '2h-2h') &
    (phase2_df1['plate'].isin(plates_of_interest))
]
# Find common mutants across plates
mutants_by_plate = {
    plate: set(subset[subset['plate'] == plate]['mutant_ID'])
    for plate in plates_of_interest
}
common_mutants = set.intersection(*mutants_by_plate.values())
# Keep only rows with common mutants
filtered_df = subset[subset['mutant_ID'].isin(common_mutants)].copy()

# Run normalization
phase2_32_2h_2h_normalized = quantile_normalize_light_regime(
    df=filtered_df,
    light_regime='2h-2h',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_32_2h_2h_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44,y2_45,y2_46,y2_47,y2_48
20096,32v1,LMJ.RY0402.039746,Cre02.g080900,A03,0.150531,0.147648,0.190585,0.162832,0.179661,0.081917,...,0.573553,0.538273,0.070297,0.073780,0.128971,0.130245,0.579163,0.563666,0.546529,0.588048
20097,32v1,LMJ.RY0402.039093,Cre13.g588959,A02,0.205374,0.175361,0.231086,0.217358,0.187040,0.160950,...,0.599761,0.621367,0.208725,0.155143,0.119765,0.146844,0.582234,0.617814,0.598952,0.601374
20098,32v1,LMJ.RY0402.068059,Cre02.g098350,P23,0.361530,0.128730,0.191766,0.304226,0.230537,0.311912,...,0.662625,0.617069,0.261803,0.091864,0.302725,0.288277,0.654173,0.588900,0.635299,0.673352
20099,32v1,LMJ.RY0402.039953,Cre02.g095137,A04,0.201794,0.120341,0.192298,0.222836,0.185538,0.188078,...,0.571886,0.605001,0.200868,0.204897,0.240325,0.205200,0.570536,0.598178,0.613911,0.581350
20100,32v1,LMJ.RY0402.211269,"Cre02.g095090,Cre01.g025600",K23,0.256748,0.183749,0.265402,0.182732,0.224207,0.133018,...,0.597203,0.592769,0.197979,0.085573,0.131479,0.082132,0.608239,0.583375,0.617401,0.601801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25596,32v2,LMJ.RY0402.168546,Cre13.g588959,K19,0.189982,0.229046,0.224126,0.236652,0.178479,0.190841,...,0.618719,0.626263,0.155895,0.189958,0.126844,0.150240,0.624704,0.627021,0.643769,0.618259
25598,32v2,LMJ.RY0402.175102,Cre10.g430200,K21,0.199567,0.189917,0.193748,0.147383,0.219911,0.217901,...,0.654129,0.661815,0.181218,0.197563,0.195471,0.168334,0.667985,0.680801,0.672917,0.659353
25599,32v2,LMJ.RY0402.176523,Cre10.g431050,K22,0.203953,0.180803,0.158158,0.184183,0.135647,0.194486,...,0.619531,0.627182,0.169204,0.084548,0.153524,0.136400,0.594731,0.616321,0.647312,0.627096
25600,32v2,LMJ.RY0402.103987,Cre01.g033450,K23,0.202092,0.128730,0.184234,0.219667,0.179796,0.147215,...,0.667737,0.672789,0.119878,0.042029,0.076034,0.090928,0.620812,0.636238,0.677753,0.669422


In [11]:
plates = ['32v1', '32v2','32v3']
phase2_32_2h_2h= phase2_df1[(phase2_df1['light_regime'] == '2h-2h') & (phase2_df1['plate'].isin(plates))].copy()
phase2_32_2h_2h[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols[:10]]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,y2_7,y2_8,y2_9,y2_10
20096,32v1,LMJ.RY0402.039746,Cre02.g080900,A03,0.151867,0.155219,0.203340,0.183596,0.200369,0.079821,0.180953,0.204326,0.489890,0.539909
20097,32v1,LMJ.RY0402.039093,Cre13.g588959,A02,0.211410,0.183080,0.246538,0.244394,0.209742,0.178042,0.244158,0.241746,0.536163,0.599595
20098,32v1,LMJ.RY0402.068059,Cre02.g098350,P23,0.368035,0.129738,0.204136,0.337703,0.261070,0.342763,0.133955,0.289914,0.608070,0.639992
20099,32v1,LMJ.RY0402.039953,Cre02.g095137,A04,0.208432,0.118789,0.204662,0.249599,0.208479,0.207087,0.253689,0.297690,0.597114,0.577680
20100,32v1,LMJ.RY0402.211269,"Cre02.g095090,Cre01.g025600",K23,0.268556,0.193137,0.286614,0.204780,0.254773,0.144287,0.212349,0.237900,0.556115,0.603011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31471,32v3,LMJ.RY0402.244803,Cre04.g224200,F04,0.338613,0.336157,0.243007,0.277811,0.296587,0.309403,0.297140,0.257629,0.636624,0.687624
31472,32v3,LMJ.RY0402.144602,Cre04.g232502,F03,0.167198,0.174993,0.158553,0.199636,0.146336,0.146424,0.210061,0.156546,0.490291,0.540711
31473,32v3,LMJ.RY0402.191571,Cre16.g667900,F02,0.267853,0.269998,0.251384,0.285739,0.315739,0.270964,0.298271,0.256781,0.619400,0.679324
31474,32v3,LMJ.RY0402.243098,Cre02.g095081,F11,0.161698,0.448613,0.441287,0.223136,0.513918,0.249503,0.655724,0.308886,0.620611,0.742572


### 32 plate 10min-10min

In [12]:
# Define inputs
plates = ['32v1', '32v2','32v3']
y2_cols = [f'y2_{i}' for i in range(1, 85)]

# Get only mutants present in all plates
plates_of_interest = plates  # or your list of plates
subset = phase2_df1[
    (phase2_df1['light_regime'] == '10min-10min') &
    (phase2_df1['plate'].isin(plates_of_interest))
]
# Find common mutants across plates
mutants_by_plate = {
    plate: set(subset[subset['plate'] == plate]['mutant_ID'])
    for plate in plates_of_interest
}
common_mutants = set.intersection(*mutants_by_plate.values())
# Keep only rows with common mutants
filtered_df = subset[subset['mutant_ID'].isin(common_mutants)].copy()

# Run normalization
phase2_32_10min_10min_normalized = quantile_normalize_light_regime(
    df=filtered_df,
    light_regime='10min-10min',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_32_10min_10min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_75,y2_76,y2_77,y2_78,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84
25979,32v2,LMJ.RY0402.057143,Cre02.g095063,P21,0.252325,0.600076,0.212981,0.616025,0.154611,0.557538,...,0.211374,0.581197,0.572785,0.141690,0.130115,0.552231,0.590618,0.173442,0.215167,0.593531
25980,32v2,LMJ.RY0402.254400,Cre03.g165000,F02,0.139602,0.553424,0.162238,0.571402,0.175931,0.575475,...,0.109352,0.544709,0.540516,0.155786,0.108542,0.549528,0.548168,0.152155,0.114758,0.567467
25981,32v2,LMJ.RY0402.068059,Cre02.g098350,F03,0.190222,0.571757,0.151024,0.573215,0.181592,0.545988,...,0.077470,0.552371,0.545465,0.127926,0.130982,0.560209,0.574736,0.152808,0.131848,0.537408
25983,32v2,LMJ.RY0402.095687,Cre12.g522700,F05,0.187657,0.634126,0.235666,0.623951,0.220177,0.622311,...,0.192883,0.632405,0.634134,0.185470,0.169306,0.645444,0.635203,0.186589,0.197138,0.646270
25984,32v2,LMJ.RY0402.046739,"Cre01.g055000,Cre15.g635350 & Cre15.g635376",F06,0.166790,0.556208,0.187365,0.586263,0.168175,0.575088,...,0.142394,0.577509,0.581763,0.155247,0.134221,0.587463,0.575475,0.135162,0.139603,0.593742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49408,32v1,LMJ.RY0402.143891,Cre03.g192550,F14,0.160223,0.610768,0.149775,0.594813,0.196688,0.623193,...,0.158364,0.589016,0.625703,0.181573,0.124541,0.597839,0.617447,0.110051,0.119934,0.608928
49409,32v1,LMJ.RY0402.040792,Cre02.g098750,A06,0.098860,0.435042,0.055831,0.428606,0.100949,0.476877,...,0.060769,0.456815,0.471819,0.046240,0.042112,0.447715,0.463021,0.061366,0.101600,0.468827
49410,32v1,LMJ.RY0402.039093,Cre13.g588959,A02,0.265926,0.601820,0.272934,0.604315,0.277045,0.612642,...,0.229069,0.573854,0.593886,0.209216,0.183525,0.583581,0.618895,0.160035,0.263910,0.620375
49411,32v1,LMJ.RY0402.039746,Cre02.g080900,A03,0.119251,0.517116,0.177985,0.554442,0.127217,0.524770,...,0.115899,0.540424,0.541722,0.155786,0.190849,0.532883,0.550226,0.173067,0.157799,0.548890


In [13]:
plates = ['32v1', '32v2','32v3']
phase2_32_10min_10min= phase2_df1[(phase2_df1['light_regime'] == '10min-10min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_32_10min_10min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_75,y2_76,y2_77,y2_78,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84
25979,32v2,LMJ.RY0402.057143,Cre02.g095063,P21,0.262980,0.600246,0.219033,0.621269,0.159767,0.556345,...,0.242956,0.576999,0.572204,0.165417,0.151368,0.550272,0.591449,0.199866,0.251105,0.597794
25980,32v2,LMJ.RY0402.254400,Cre03.g165000,F02,0.133935,0.549760,0.168523,0.572130,0.181149,0.573647,...,0.120760,0.541412,0.535098,0.182445,0.126203,0.547895,0.545176,0.175703,0.134795,0.567235
25981,32v2,LMJ.RY0402.068059,Cre02.g098350,F03,0.189527,0.570591,0.155602,0.574427,0.187456,0.543369,...,0.084384,0.547923,0.541131,0.149516,0.152329,0.558927,0.573419,0.176161,0.156136,0.535751
25982,32v2,LMJ.RY0402.097699,Cre03.g191850,F04,0.214386,0.658445,0.293199,0.661686,0.280986,0.677720,...,0.237767,0.661483,0.677007,0.288347,0.228170,0.661856,0.667374,0.289336,0.261078,0.676675
25983,32v2,LMJ.RY0402.095687,Cre12.g522700,F05,0.186603,0.640955,0.241673,0.630025,0.235579,0.628733,...,0.221771,0.639184,0.644066,0.217143,0.194892,0.657078,0.643909,0.214087,0.229530,0.657367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49408,32v1,LMJ.RY0402.143891,Cre03.g192550,F14,0.154998,0.613843,0.140420,0.595764,0.170748,0.613319,...,0.116573,0.591900,0.611659,0.126265,0.091282,0.596417,0.606597,0.081601,0.091056,0.605123
49409,32v1,LMJ.RY0402.040792,Cre02.g098750,A06,0.111104,0.482539,0.085763,0.466140,0.096964,0.508670,...,0.037464,0.490062,0.498594,0.021862,0.029740,0.502705,0.490336,0.040178,0.072959,0.497550
49410,32v1,LMJ.RY0402.039093,Cre13.g588959,A02,0.219566,0.607292,0.219560,0.600911,0.220074,0.607097,...,0.157615,0.582513,0.597039,0.140630,0.130838,0.588998,0.607419,0.116089,0.184735,0.612898
49411,32v1,LMJ.RY0402.039746,Cre02.g080900,A03,0.127561,0.542836,0.160912,0.566810,0.122559,0.546330,...,0.090818,0.555090,0.560084,0.110369,0.134525,0.548479,0.564581,0.125281,0.112826,0.563691


### plate 32 1min-1min

In [14]:
# Define inputs
plates = ['32v1', '32v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]

# Run normalization
phase2_32_1min_1min_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='1min-1min',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_32_1min_1min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
18965,32v1,LMJ.RY0402.039953,Cre02.g095137,A04,0.241880,0.543796,0.241451,0.521001,0.188137,0.532939,...,0.177955,0.512507,0.206511,0.481452,0.152056,0.506778,0.149522,0.500055,0.187364,0.489872
18966,32v1,LMJ.RY0402.140370,Cre07.g325754,F04,0.221190,0.537437,0.203070,0.532979,0.193760,0.550000,...,0.172401,0.550313,0.168994,0.512607,0.210154,0.512171,0.181395,0.513655,0.163450,0.525887
18967,32v1,LMJ.RY0402.141149,Cre09.g396028,F05,0.248423,0.557271,0.217309,0.547515,0.203251,0.533754,...,0.200336,0.528022,0.202822,0.522878,0.206546,0.516255,0.192817,0.515627,0.197556,0.528106
18968,32v1,LMJ.RY0402.141182,Cre01.g012150,F06,0.314872,0.633194,0.311100,0.617057,0.290201,0.611051,...,0.229812,0.574785,0.229946,0.589528,0.244232,0.575281,0.219532,0.570434,0.206438,0.590196
18969,32v1,LMJ.RY0402.141305,Cre10.g461700,F07,0.088653,0.381749,0.063845,0.335754,0.045164,0.327650,...,-0.040549,0.265736,-0.015340,0.294519,-0.027927,0.228526,-0.033572,0.231570,-0.045165,0.270235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30343,32v3,LMJ.RY0402.215150,Cre17.g724650,K21,0.214869,0.517393,0.192402,0.503079,0.161449,0.520058,...,0.159316,0.532841,0.151976,0.502543,0.156970,0.449831,0.128560,0.516609,0.162348,0.523440
30344,32v3,LMJ.RY0402.173502,Cre06.g278219,K22,0.188189,0.575430,0.216061,0.534907,0.181785,0.525954,...,0.164615,0.517748,0.186050,0.506822,0.182558,0.505700,0.180329,0.496058,0.175098,0.514159
30345,32v3,LMJ.RY0402.077964,Cre01.g019550,K23,0.096446,0.391698,0.108129,0.383888,0.075375,0.344868,...,-0.023346,0.334288,-0.041153,0.294519,0.011200,0.356563,0.032186,0.326675,0.020075,0.325974
30346,32v3,LMJ.RY0402.107867,Cre17.g704400,K24,0.208007,0.547268,0.163092,0.481765,0.139037,0.496530,...,0.147033,0.495498,0.171837,0.478039,0.094385,0.480821,0.122058,0.496682,0.108926,0.478176


In [15]:
plates = ['32v1', '32v2','32v3']
phase2_32_1min_1min= phase2_df1[(phase2_df1['light_regime'] == '1min-1min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_32_1min_1min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
18965,32v1,LMJ.RY0402.039953,Cre02.g095137,A04,0.233858,0.537572,0.229277,0.512299,0.169666,0.523894,...,0.161174,0.497308,0.194471,0.466288,0.134733,0.497406,0.136601,0.488689,0.173369,0.479915
18966,32v1,LMJ.RY0402.140370,Cre07.g325754,F04,0.214133,0.530224,0.187723,0.524317,0.174424,0.539930,...,0.155270,0.539173,0.154141,0.500816,0.197577,0.503238,0.167032,0.504599,0.147335,0.519615
18967,32v1,LMJ.RY0402.141149,Cre09.g396028,F05,0.242308,0.550901,0.202172,0.537514,0.186665,0.524460,...,0.183980,0.515634,0.189783,0.511833,0.192696,0.506405,0.177764,0.507517,0.182437,0.522160
18968,32v1,LMJ.RY0402.141182,Cre01.g012150,F06,0.309315,0.630056,0.293818,0.611724,0.277047,0.605010,...,0.215312,0.568053,0.221272,0.580931,0.236120,0.569039,0.210581,0.567994,0.192846,0.586325
18969,32v1,LMJ.RY0402.141305,Cre10.g461700,F07,0.078567,0.358015,0.035405,0.306229,0.030843,0.302581,...,-0.068828,0.216017,-0.037485,0.252459,-0.060020,0.199548,-0.047861,0.201388,-0.053832,0.251400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30343,32v3,LMJ.RY0402.215150,Cre17.g724650,K21,0.222528,0.524419,0.208723,0.515534,0.179963,0.529065,...,0.176478,0.543358,0.167147,0.515697,0.173438,0.462974,0.144843,0.524365,0.179344,0.530208
30344,32v3,LMJ.RY0402.173502,Cre06.g278219,K22,0.195087,0.578093,0.231456,0.544763,0.201840,0.534820,...,0.181478,0.530981,0.198928,0.518615,0.196738,0.514623,0.195312,0.508109,0.190457,0.520550
30345,32v3,LMJ.RY0402.077964,Cre01.g019550,K23,0.105385,0.410684,0.125441,0.409402,0.094088,0.360150,...,0.001686,0.347981,-0.015515,0.336580,0.037797,0.357349,0.044077,0.356539,0.029723,0.341814
30346,32v3,LMJ.RY0402.107867,Cre17.g704400,K24,0.217012,0.553343,0.178930,0.494776,0.157174,0.508011,...,0.165725,0.511446,0.186329,0.493573,0.112441,0.493025,0.137365,0.509040,0.130978,0.489089


### plate 32 30s 30s

In [16]:
# Define inputs
plates = ['32v1','32v2', '32v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]

# Get only mutants present in all plates
plates_of_interest = plates  # or your list of plates
subset = phase2_df1[
    (phase2_df1['light_regime'] == '30s-30s') &
    (phase2_df1['plate'].isin(plates_of_interest))
]
# Find common mutants across plates
mutants_by_plate = {
    plate: set(subset[subset['plate'] == plate]['mutant_ID'])
    for plate in plates_of_interest
}
common_mutants = set.intersection(*mutants_by_plate.values())
# Keep only rows with common mutants
filtered_df = subset[subset['mutant_ID'].isin(common_mutants)].copy()

# Run normalization
phase2_32_30s_30s_normalized = quantile_normalize_light_regime(
    df=filtered_df,
    light_regime='30s-30s',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_32_30s_30s_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
20465,32v1,LMJ.RY0402.178298,Cre17.g706600,I01,0.236108,0.582230,0.217537,0.566867,0.265125,0.548024,...,0.144339,0.516578,0.178357,0.494094,0.146091,0.502648,0.169381,0.512597,0.185346,0.516117
20466,32v1,LMJ.RY0402.039746,Cre02.g080900,A03,0.131206,0.488388,0.107172,0.504486,0.213122,0.461350,...,0.169320,0.468461,0.148803,0.468926,0.129697,0.456855,0.122132,0.429651,0.182418,0.443441
20467,32v1,LMJ.RY0402.141149,Cre09.g396028,F05,0.282725,0.559068,0.279490,0.546744,0.253069,0.530929,...,0.258087,0.533187,0.260240,0.535280,0.247823,0.521441,0.260362,0.540684,0.254506,0.520623
20468,32v1,LMJ.RY0402.141182,Cre01.g012150,F06,0.385324,0.652580,0.344122,0.636246,0.367476,0.626307,...,0.309683,0.590474,0.282534,0.607514,0.288555,0.589387,0.273757,0.597456,0.274664,0.583039
20469,32v1,LMJ.RY0402.141305,Cre10.g461700,F07,0.101250,0.328742,0.124758,0.363125,0.084927,0.350393,...,0.023629,0.264849,0.038083,0.272419,0.011944,0.285665,0.025325,0.307037,0.043352,0.260151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31846,32v3,LMJ.RY0402.172376,Cre06.g278219,K17,0.162838,0.478979,0.172132,0.458405,0.098434,0.462563,...,0.133059,0.443732,0.144684,0.454231,0.084734,0.433913,0.080896,0.464599,0.129972,0.434808
31847,32v3,LMJ.RY0402.253374,Cre02.g077951,K18,0.279874,0.580299,0.247797,0.581627,0.247644,0.533488,...,0.237425,0.531861,0.220803,0.548738,0.267460,0.544917,0.219499,0.533287,0.234100,0.547163
31848,32v3,LMJ.RY0402.142353,Cre06.g295050,K19,0.370358,0.610812,0.342778,0.608556,0.310536,0.619794,...,0.258087,0.536016,0.214504,0.533467,0.264688,0.514783,0.248324,0.519705,0.249424,0.565206
31849,32v3,LMJ.RY0402.111037,Cre13.g607300,K20,0.375334,0.633463,0.363581,0.616558,0.386362,0.624330,...,0.330839,0.553445,0.293426,0.528481,0.263579,0.542947,0.273757,0.539980,0.288055,0.531311


In [17]:
plates = ['32v1', '32v2','32v3']
phase2_32_1min_1min= phase2_df1[(phase2_df1['light_regime'] == '1min-1min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_32_1min_1min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
18965,32v1,LMJ.RY0402.039953,Cre02.g095137,A04,0.233858,0.537572,0.229277,0.512299,0.169666,0.523894,...,0.161174,0.497308,0.194471,0.466288,0.134733,0.497406,0.136601,0.488689,0.173369,0.479915
18966,32v1,LMJ.RY0402.140370,Cre07.g325754,F04,0.214133,0.530224,0.187723,0.524317,0.174424,0.539930,...,0.155270,0.539173,0.154141,0.500816,0.197577,0.503238,0.167032,0.504599,0.147335,0.519615
18967,32v1,LMJ.RY0402.141149,Cre09.g396028,F05,0.242308,0.550901,0.202172,0.537514,0.186665,0.524460,...,0.183980,0.515634,0.189783,0.511833,0.192696,0.506405,0.177764,0.507517,0.182437,0.522160
18968,32v1,LMJ.RY0402.141182,Cre01.g012150,F06,0.309315,0.630056,0.293818,0.611724,0.277047,0.605010,...,0.215312,0.568053,0.221272,0.580931,0.236120,0.569039,0.210581,0.567994,0.192846,0.586325
18969,32v1,LMJ.RY0402.141305,Cre10.g461700,F07,0.078567,0.358015,0.035405,0.306229,0.030843,0.302581,...,-0.068828,0.216017,-0.037485,0.252459,-0.060020,0.199548,-0.047861,0.201388,-0.053832,0.251400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30343,32v3,LMJ.RY0402.215150,Cre17.g724650,K21,0.222528,0.524419,0.208723,0.515534,0.179963,0.529065,...,0.176478,0.543358,0.167147,0.515697,0.173438,0.462974,0.144843,0.524365,0.179344,0.530208
30344,32v3,LMJ.RY0402.173502,Cre06.g278219,K22,0.195087,0.578093,0.231456,0.544763,0.201840,0.534820,...,0.181478,0.530981,0.198928,0.518615,0.196738,0.514623,0.195312,0.508109,0.190457,0.520550
30345,32v3,LMJ.RY0402.077964,Cre01.g019550,K23,0.105385,0.410684,0.125441,0.409402,0.094088,0.360150,...,0.001686,0.347981,-0.015515,0.336580,0.037797,0.357349,0.044077,0.356539,0.029723,0.341814
30346,32v3,LMJ.RY0402.107867,Cre17.g704400,K24,0.217012,0.553343,0.178930,0.494776,0.157174,0.508011,...,0.165725,0.511446,0.186329,0.493573,0.112441,0.493025,0.137365,0.509040,0.130978,0.489089


## plate 32 5min-5min

In [ ]:
# Define inputs
plates = ['32v1', '32v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]


plates_of_interest = plates  # or your list of plates
subset = phase2_df1[
    (phase2_df1['light_regime'] == '5min-5min') &
    (phase2_df1['plate'].isin(plates_of_interest))
]
mutants_by_plate = {
    plate: set(subset[subset['plate'] == plate]['mutant_ID'])
    for plate in plates_of_interest
}
common_mutants = set.intersection(*mutants_by_plate.values())
filtered_df = subset[subset['mutant_ID'].isin(common_mutants)].copy()

# Run normalization
phase2_32_5min_5min_normalized = quantile_normalize_light_regime(
    df=filtered_df,
    light_regime='5min-5min',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_32_5min_5min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
21214,32v1,LMJ.RY0402.039746,Cre02.g080900,A03,0.066235,0.512625,0.127016,0.445220,0.133946,0.479517,...,0.121555,0.493945,0.088316,0.475941,0.091372,0.498231,0.078847,0.482626,0.072115,0.430700
21215,32v1,LMJ.RY0402.068059,Cre02.g098350,P23,0.206112,0.517856,0.213473,0.503142,0.103303,0.514132,...,0.128628,0.495969,0.115154,0.516992,0.112987,0.516191,0.160749,0.497494,0.108108,0.491651
21216,32v1,LMJ.RY0402.140370,Cre07.g325754,F04,0.126899,0.588630,0.185261,0.575672,0.140589,0.546956,...,0.158628,0.560728,0.135355,0.550825,0.131463,0.556096,0.154141,0.519992,0.159889,0.539162
21217,32v1,LMJ.RY0402.141149,Cre09.g396028,F05,0.212117,0.573649,0.191987,0.558713,0.201548,0.559001,...,0.206353,0.574806,0.191375,0.564950,0.171352,0.556904,0.184727,0.572343,0.171462,0.565114
21218,32v1,LMJ.RY0402.141182,Cre01.g012150,F06,0.298563,0.685677,0.271249,0.646552,0.269846,0.647872,...,0.221376,0.634736,0.203159,0.643354,0.176729,0.638047,0.185846,0.624856,0.189115,0.651545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32976,32v3,LMJ.RY0402.172376,Cre06.g278219,K17,0.161665,0.546886,0.152123,0.546525,0.139263,0.501174,...,0.106425,0.520888,0.118845,0.528059,0.079035,0.525189,0.097796,0.514884,0.128793,0.501367
32977,32v3,LMJ.RY0402.253374,Cre02.g077951,K18,0.211904,0.618969,0.225580,0.604974,0.218748,0.617002,...,0.182199,0.607109,0.192056,0.588480,0.188561,0.609928,0.189686,0.602563,0.175053,0.588360
32978,32v3,LMJ.RY0402.142353,Cre06.g295050,K19,0.333872,0.651408,0.306870,0.628447,0.284727,0.628244,...,0.234213,0.581353,0.191810,0.602024,0.187656,0.613669,0.186428,0.576298,0.197069,0.590519
32979,32v3,LMJ.RY0402.111037,Cre13.g607300,K20,0.350555,0.660465,0.290047,0.662278,0.306150,0.635914,...,0.260002,0.598162,0.231888,0.599515,0.222821,0.624601,0.231462,0.589404,0.199382,0.600417


In [19]:
plates = ['32v1', '32v2','32v3']
phase2_32_5min_5min= phase2_df1[(phase2_df1['light_regime'] == '5min-5min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_32_5min_5min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
21214,32v1,LMJ.RY0402.039746,Cre02.g080900,A03,0.064784,0.515743,0.125836,0.461769,0.129919,0.486931,...,0.115525,0.487931,0.087765,0.478022,0.089186,0.495418,0.078291,0.482975,0.075634,0.441132
21215,32v1,LMJ.RY0402.068059,Cre02.g098350,P23,0.199220,0.520778,0.205126,0.512809,0.104711,0.515894,...,0.122010,0.489515,0.111999,0.516971,0.110752,0.512499,0.155199,0.498247,0.106405,0.491521
21216,32v1,LMJ.RY0402.140370,Cre07.g325754,F04,0.128917,0.587394,0.176464,0.577782,0.135835,0.549125,...,0.149161,0.550927,0.128568,0.548255,0.126675,0.548721,0.150912,0.518071,0.154440,0.537140
21217,32v1,LMJ.RY0402.141149,Cre09.g396028,F05,0.207181,0.573921,0.183777,0.563610,0.191069,0.559982,...,0.191554,0.565608,0.183616,0.561258,0.165880,0.549255,0.178905,0.569218,0.166721,0.562313
21218,32v1,LMJ.RY0402.141182,Cre01.g012150,F06,0.295172,0.683738,0.259389,0.650068,0.262381,0.648728,...,0.207142,0.629505,0.193984,0.639168,0.170363,0.633652,0.180482,0.621444,0.183000,0.646111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32976,32v3,LMJ.RY0402.172376,Cre06.g278219,K17,0.165143,0.546715,0.155068,0.539573,0.143136,0.494826,...,0.110689,0.525424,0.122102,0.528487,0.079885,0.529552,0.102669,0.514400,0.131785,0.502319
32977,32v3,LMJ.RY0402.253374,Cre02.g077951,K18,0.216786,0.620412,0.231849,0.600391,0.229748,0.621471,...,0.193271,0.616336,0.199966,0.593269,0.194552,0.618953,0.194994,0.603270,0.181060,0.591992
32978,32v3,LMJ.RY0402.142353,Cre06.g295050,K19,0.336214,0.649167,0.318989,0.624929,0.292057,0.630202,...,0.246055,0.588761,0.199707,0.606286,0.193395,0.620690,0.191392,0.578797,0.202505,0.592883
32979,32v3,LMJ.RY0402.111037,Cre13.g607300,K20,0.357303,0.661348,0.301797,0.658366,0.313330,0.636542,...,0.274204,0.606875,0.238273,0.603505,0.233116,0.629246,0.241554,0.590817,0.205434,0.602844


### plate 32 1min-5min

In [20]:
# Define inputs
plates = ['32v2', '32v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]

# Run normalization
phase2_32_1min_5min_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='1min-5min',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_32_1min_5min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
26356,32v2,LMJ.RY0402.191571,Cre16.g667900,P19,0.323898,0.655719,0.230521,0.643118,0.294171,0.660975,...,0.231634,0.628760,0.210645,0.640874,0.226525,0.633155,0.195313,0.627878,0.241295,0.601422
26357,32v2,LMJ.RY0402.243149,Cre16.g688350,P20,0.345886,0.671111,0.334762,0.666956,0.357624,0.666256,...,0.258061,0.647701,0.243886,0.629498,0.268779,0.638416,0.228452,0.637112,0.231213,0.640981
26358,32v2,LMJ.RY0402.055870,Cre06.g259950,D23,0.066960,0.550199,0.108798,0.489846,0.089321,0.524705,...,0.097055,0.510789,-0.064588,0.496999,0.045698,0.464235,0.012856,0.484492,0.044320,0.488435
26359,32v2,LMJ.RY0402.177454,Cre03.g205428,E24,0.148632,0.542387,0.163560,0.542097,0.148222,0.512292,...,0.111103,0.553120,0.082649,0.530045,0.087215,0.509031,0.071062,0.525345,0.094786,0.531931
26360,32v2,LMJ.RY0402.064788,Cre04.g222600,F01,0.177765,0.573568,0.176273,0.566595,0.106406,0.538277,...,0.143012,0.524573,0.086563,0.529301,0.093945,0.565497,0.063002,0.527512,0.094848,0.547865
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32600,32v3,LMJ.RY0402.173502,Cre06.g278219,K22,0.277002,0.627887,0.200316,0.593957,0.213173,0.606533,...,0.177382,0.623317,0.172689,0.598666,0.187393,0.602917,0.212530,0.578168,0.148811,0.584973
32601,32v3,LMJ.RY0402.077964,Cre01.g019550,K23,0.094580,0.495167,0.060049,0.507656,0.105288,0.472075,...,0.027043,0.461673,0.075572,0.456686,0.074289,0.478199,0.073373,0.443533,-0.023836,0.483576
32602,32v3,LMJ.RY0402.107867,Cre17.g704400,K24,0.203005,0.581370,0.228428,0.588547,0.159388,0.560500,...,0.086908,0.541080,0.117544,0.542755,0.121992,0.561828,0.167308,0.552454,0.126462,0.547865
32603,32v3,LMJ.RY0402.104944,Cre14.g626350,K15,0.196969,0.555637,0.208967,0.548035,0.166834,0.546243,...,0.122105,0.544206,0.144620,0.516438,0.153112,0.554247,0.122562,0.527512,0.119176,0.559035


In [21]:
plates = ['32v1','32v2','32v3']
phase2_32_1min_5min= phase2_df1[(phase2_df1['light_regime'] == '1min-5min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_32_1min_5min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
20839,32v1,LMJ.RY0402.065081,Cre12.g493050,P21,0.361300,0.683018,0.322445,0.653674,0.289277,0.656564,...,0.230841,0.638294,0.205548,0.651578,0.255440,0.642147,0.210781,0.647060,0.234306,0.639355
20840,32v1,LMJ.RY0402.146061,Cre02.g115600,P24,0.113216,0.550084,0.153382,0.534856,0.126855,0.482418,...,0.159373,0.543407,0.122998,0.477443,0.123336,0.483387,0.036329,0.538936,0.124851,0.486202
20841,32v1,LMJ.RY0402.140370,Cre07.g325754,F04,0.210440,0.609993,0.192245,0.569583,0.182570,0.577755,...,0.152772,0.580935,0.171840,0.595250,0.172512,0.592476,0.120154,0.576039,0.163163,0.594737
20842,32v1,LMJ.RY0402.141149,Cre09.g396028,F05,0.244390,0.619670,0.233768,0.603004,0.199422,0.582777,...,0.209583,0.607906,0.197206,0.612862,0.209954,0.606830,0.195620,0.612092,0.194821,0.624483
20843,32v1,LMJ.RY0402.141182,Cre01.g012150,F06,0.354000,0.714778,0.347718,0.691529,0.311295,0.683463,...,0.219159,0.680012,0.243303,0.685899,0.266637,0.666464,0.262113,0.666102,0.243847,0.681892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32600,32v3,LMJ.RY0402.173502,Cre06.g278219,K22,0.273565,0.624527,0.194420,0.588244,0.209162,0.604629,...,0.175016,0.622051,0.163439,0.598140,0.182584,0.601016,0.203296,0.573038,0.139926,0.582610
32601,32v3,LMJ.RY0402.077964,Cre01.g019550,K23,0.084776,0.489463,0.062597,0.500775,0.094853,0.479271,...,0.025174,0.458077,0.064198,0.454603,0.062432,0.483878,0.062370,0.441900,-0.025002,0.481994
32602,32v3,LMJ.RY0402.107867,Cre17.g704400,K24,0.196489,0.577164,0.223327,0.582785,0.155346,0.559474,...,0.085081,0.538116,0.106573,0.543370,0.115722,0.559473,0.159333,0.550606,0.117841,0.545491
32603,32v3,LMJ.RY0402.104944,Cre14.g626350,K15,0.189363,0.551925,0.203304,0.537722,0.162288,0.543818,...,0.120343,0.541137,0.135613,0.517293,0.144793,0.551168,0.111528,0.520736,0.110785,0.556165


In [22]:
phase2_32_quantile1= pd.concat([
    phase2_32_20h_ML_normalized,
    phase2_32_20h_HL_normalized,
    phase2_32_2h_2h_normalized,
    phase2_32_10min_10min_normalized,
    phase2_32_1min_1min_normalized,
    phase2_32_30s_30s_normalized,
    phase2_32_5min_5min_normalized,
    phase2_32_1min_5min_normalized
], ignore_index=True)

In [23]:
phase2_32_quantile1.to_csv('phase2_32_quantile1.csv',index=False)

In [24]:
phase2_32_quantile1.shape

(7496, 467)

In [25]:
plates = ['32v1', '32v2','32v3']
data=phase2_df1[phase2_df1['plate'].isin(plates)]
data.shape

(9031, 467)